In [49]:
import argparse
import datetime
import math
import os
from time import time
from typing import Any, List, Tuple

import mlflow
import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed
from sparsemax import Sparsemax

from artifacts import (
    build_plot_df_wrapper,
    save_csv_artifact,
    save_plot_strategy,
)
from BoTorchOptimizer import BoTorchOptimizer
from data import (
    apply_final_treatment,
    join_metadata,
    load_metadata_artefacts,
    load_odds,
)
from dependencies.config import load_config
from dependencies.utils import get_bet_return, save_df_as_parquet, softmax, sparsemax
from filter import filter_by_linear_combination
from GameProbs import GameProbs

config = load_config("config/config.yml")

sparsemax = Sparsemax(dim=-1)

from loguru import logger


In [50]:

def setup(args):
    metadata, gameid_to_outcome = load_metadata_artefacts(config.metadata_path)
    odds = load_odds(config.odds_path, args.bookmakers)
    print(odds.shape)
    odds = join_metadata(odds, metadata)

    odds = odds.sort_values(["Datetime", "GameId"], ascending=True)

    #odds = odds[(odds.Datetime.apply(str)>"2019-08-01")&(odds.Datetime.apply(str)<"2019-09-01")]
    #odds = odds[(odds.Datetime.apply(str)>"2019-01-01")&(odds.Datetime.apply(str)<"2020-01-01")]
    odds = odds[(odds.Datetime.apply(str)>"2023-07-21")]
    
    # odds = odds[
    #     (odds.Datetime.apply(str) > "2019-06-01")
    #     & (odds.Datetime.apply(str) < "2019-07-01")
    # ]

    # odds = odds[odds]

    return odds, gameid_to_outcome


def process_group(
    group: Tuple[str, pd.DataFrame], gameid_to_outcome, args
) -> List[List[Any]]:
    is_valid_solution = True

    date, group_data = group

    games_ids = group_data["GameId"].unique()

    # Initialize dict to store dataframes of favorable bet opportunities
    odds_dict = {}
    # Initialize dict to store 7x7 matrices/dataframes of real probabilities
    df_probs_dict = {}

    if len(games_ids) > args.min_games:
        for game_id in games_ids:
            df = GameProbs(game_id).build_dataframe()

            odds_sample = group_data[(group_data.GameId == game_id)]
            odds_sample = apply_final_treatment(df_odds=odds_sample, df_real_prob=df)
            if not args.do_baseline:
                odds_sample = filter_by_linear_combination(odds_sample, n=args.bets_per_game, weight=args.weight)
            else:
                odds_sample = odds_sample.sample(1)
            odds_dict[game_id] = odds_sample
            df_probs_dict[game_id] = df

        odds_dt = pd.concat(odds_dict.values())

        if len(odds_dt) <= config.max_vector_length and len(odds_dt) > 1:
        #if len(odds_dt) <= config.max_vector_length :
            iteration_date = odds_dt.Datetime.apply(str).unique()[0]
            print(f"Date: {iteration_date}")

            odds_favorable = torch.tensor(np.array(odds_dt["Odd"]))
            real_prob_favorable = torch.tensor(np.array(odds_dt["real_prob"]))
            event_favorable = list(odds_dt["BetMap"].values)
            games_ids = np.array(odds_dt["GameId"])
            time_limit_flag = None

            if not args.do_baseline:
                # try:
                print("Execution of minimization task...")


                optimizer_instance = BoTorchOptimizer(
                    n_iterations=args.n_iterations,
                    public_odd=odds_favorable,
                    real_probabilities=real_prob_favorable,
                    event=event_favorable,
                    games_ids=games_ids,
                    df_probs_dict=df_probs_dict,
                )

                solution = optimizer_instance.run_optimization()

                print("Finalization of minimization task...")

                # except ValueError:
                # continue

                if any(math.isnan(x) for x in solution):
                    is_valid_solution = False
                odds_dt["solution"] = sparsemax(torch.tensor(np.array([solution]))).tolist()[0]
                #odds_dt["solution"] = softmax(solution)

            else:
                odds_dt["solution"] = 1

            save_df_as_parquet(odds_dt, str(date))

            track_record = []

            for game_id, game_data in odds_dt.groupby("GameId", sort=False):
                scenario = gameid_to_outcome[game_id]
                financial_return = get_bet_return(
                    df=game_data, allocation_array=game_data.solution, scenario=scenario
                )

                print(
                    f"game_id: {game_id}; financial_return: {np.round(financial_return, 3)}"
                )

                track_record.append(
                    [
                        str(game_id),
                        financial_return,
                        len(game_data),
                        odds_dt.n_favorable_bets.values[0],
                        time_limit_flag,
                        is_valid_solution,
                        iteration_date,
                    ]
                )

            return track_record

In [51]:
#args = parser.parse_args()
parser = argparse.ArgumentParser()
parser.add_argument(
    '--bookmakers',
    nargs='+',
    default=None,
    help='A list of strings',
)
parser.add_argument(
    "--aggregator", type=str, help="aggregate by GameId or by Datetime"
)
parser.add_argument(
    "--min_games",
    type=int,
    default=0,
    help="threshold of minimum number of games to enter the optimization task",
)
parser.add_argument(
    "--bets_per_game", type=int, default=5, help="number of bets per game"
)
parser.add_argument(
    "--weight",
    type=float,
    default=0.5,
    help="weight of the linear combination filter",
)
parser.add_argument(
    "--n_iterations",
    type=int,
    default=10,
    help="number of iterations to run the optimization task",
)
parser.add_argument(
    "--do_baseline",
    action="store_true",
    help="flag to apply baseline logic or not, not specifying the argument return the opposite of the action",
)
parser.add_argument(
    "--n_jobs",
    type=int,
    default=1,
    help="number of jobs to run in parallel",
)
parser.add_argument(
    "--save_experiment",
    action="store_true",
    help="flag to save the experiment artefacts, not specifying the argument return the opposite of the action",
)
args = parser.parse_args([
"--aggregator", "Datetime",
"--min_games", "1",
"--n_iterations", "100",
#"--do_baseline", "False"
#"--save_experiment"
])
print(args)

odds, gameid_to_outcome = setup(args)

grouped = odds.groupby(args.aggregator)

Namespace(bookmakers=None, aggregator='Datetime', min_games=1, bets_per_game=5, weight=0.5, n_iterations=100, do_baseline=False, n_jobs=1, save_experiment=False)
(2113785, 7)


In [52]:
for a, b in grouped:
    print(a)
    print(b)

2023-07-22
          GameId     Sportsbook Market Scenario   Bet     Odd  public_prob  \
2078599  7290215      Stake.com    h2h     None  home    1.53     0.653595   
2078600  7290215      Stake.com    h2h     None  draw    3.70     0.270270   
2078601  7290215      Stake.com    h2h     None  away    6.20     0.161290   
2078602  7290215  BC.Game Sport    h2h     None  home    1.55     0.645161   
2078603  7290215  BC.Game Sport    h2h     None  draw    4.10     0.243902   
...          ...            ...    ...      ...   ...     ...          ...   
2014928  7290247          1xBet  exact   10 : 6   odd  100.00     0.010000   
2014929  7290247          1xBet  exact   10 : 7   odd  100.00     0.010000   
2014930  7290247          1xBet  exact   10 : 8   odd  100.00     0.010000   
2014931  7290247          1xBet  exact   10 : 9   odd  100.00     0.010000   
2014932  7290247          1xBet  exact  10 : 10   odd  100.00     0.010000   

              Home        Away    Datetime  
2078599

In [53]:
# Function to get the n-th group and its DataFrame
def get_nth_group(grouped, n):
    for i, (group_key, group_df) in enumerate(grouped):
        if i == n:
            return group_key, group_df
    raise IndexError("Group index out of range")
n = 0
group = get_nth_group(grouped, n)

In [54]:
date, group_data = group
date

datetime.date(2023, 7, 22)

In [55]:

games_ids = group_data["GameId"].unique()

# Initialize dict to store dataframes of favorable bet opportunities
odds_dict = {}
# Initialize dict to store 7x7 matrices/dataframes of real probabilities
df_probs_dict = {}

if len(games_ids) > args.min_games:
    for game_id in games_ids:
        df = GameProbs(game_id).build_dataframe()

        odds_sample = group_data[(group_data.GameId == game_id)]
        odds_sample = apply_final_treatment(df_odds=odds_sample, df_real_prob=df)
        if not args.do_baseline:
            odds_sample = filter_by_linear_combination(odds_sample, n=args.bets_per_game, weight=args.weight)
        else:
            odds_sample = odds_sample.sample(1)
        odds_dict[game_id] = odds_sample
        df_probs_dict[game_id] = df

    odds_dt = pd.concat(odds_dict.values())

    if len(odds_dt) <= config.max_vector_length and len(odds_dt) > 1:
    #if len(odds_dt) <= config.max_vector_length :
        iteration_date = odds_dt.Datetime.apply(str).unique()[0]
        print(f"Date: {iteration_date}")

        odds_favorable = torch.tensor(np.array(odds_dt["Odd"]))
        real_prob_favorable = torch.tensor(np.array(odds_dt["real_prob"]))
        event_favorable = list(odds_dt["BetMap"].values)
        games_ids = np.array(odds_dt["GameId"])
        time_limit_flag = None

        if not args.do_baseline:
            # try:
            print("Execution of minimization task...")


            optimizer_instance = BoTorchOptimizer(
                n_iterations=args.n_iterations,
                public_odd=odds_favorable,
                real_probabilities=real_prob_favorable,
                event=event_favorable,
                games_ids=games_ids,
                df_probs_dict=df_probs_dict,
            )

            solution = optimizer_instance.run_optimization()

            print("Finalization of minimization task...")

            # except ValueError:
            # continue

            if any(math.isnan(x) for x in solution):
                is_valid_solution = False
            odds_dt["solution"] = sparsemax(torch.tensor(np.array([solution]))).tolist()[0]
            #odds_dt["solution"] = softmax(solution)

        else:
            odds_dt["solution"] = 1

        track_record = []
            
        financial_return_aggregated = 0

        for game_id, game_data in odds_dt.groupby("GameId", sort=False):
            scenario = gameid_to_outcome[game_id]
            financial_return = get_bet_return(
                df=game_data, allocation_array=game_data.solution, scenario=scenario
            )
            financial_return_aggregated += financial_return
            logger.info(
                f"game_id: {game_id}; financial_return: {np.round(financial_return, 3)}"
            )

Date: 2023-07-22
Execution of minimization task...
New best value found: tensor([[1.9445]], dtype=torch.float64)
New best value found: tensor([[1.9679]], dtype=torch.float64)
New best value found: tensor([[1.9714]], dtype=torch.float64)
New best value found: tensor([[1.9951]], dtype=torch.float64)
New best value found: tensor([[2.0018]], dtype=torch.float64)
New best value found: tensor([[2.0247]], dtype=torch.float64)
New best value found: tensor([[2.0417]], dtype=torch.float64)
New best value found: tensor([[2.0566]], dtype=torch.float64)
New best value found: tensor([[2.0615]], dtype=torch.float64)
New best value found: tensor([[2.0704]], dtype=torch.float64)
New best value found: tensor([[2.0847]], dtype=torch.float64)
New best value found: tensor([[2.0937]], dtype=torch.float64)
New best value found: tensor([[2.0974]], dtype=torch.float64)
New best value found: tensor([[2.1066]], dtype=torch.float64)
New best value found: tensor([[2.1123]], dtype=torch.float64)
New best value foun

2024-07-04 21:38:21.104 | INFO     | __main__:<module>:73 - game_id: 7290215; financial_return: 0.818
2024-07-04 21:38:21.107 | INFO     | __main__:<module>:73 - game_id: 7290221; financial_return: 0.0
2024-07-04 21:38:21.110 | INFO     | __main__:<module>:73 - game_id: 7290222; financial_return: 1.085
2024-07-04 21:38:21.114 | INFO     | __main__:<module>:73 - game_id: 7290247; financial_return: 0.0


Finalization of minimization task...


In [8]:
odds_dt

,GameId,Sportsbook,Market,Scenario,Bet,Odd,public_prob,Home,Away,Datetime,BetMap,real_prob,bet_flag,n_favorable_bets,odd_dist,score,expected_return,solution
6,7290215,Fezbet,over/under,4.5,over,5.50,0.181818,Palmeiras,Fortaleza,2023-07-22,"[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, ...",0.5222,True,13,0.7,0.61110,2.872100,0.000000
4,7290215,Stake.com,over/under,3.5,over,3.15,0.317460,Palmeiras,Fortaleza,2023-07-22,"[0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, ...",0.6586,True,13,0.5,0.57930,2.074590,0.254973
8,7290215,Fezbet,over/under,5.5,over,9.00,0.111111,Palmeiras,Fortaleza,2023-07-22,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, ...",0.4119,True,13,0.7,0.55595,3.707100,0.000000
20,7290215,1xBet,spread,-3/+3,home,10.00,0.100000,Palmeiras,Fortaleza,2023-07-22,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.3332,True,13,0.7,0.51660,3.332000,0.000000
10,7290215,1xBet,over/under,6.5,over,26.00,0.038462,Palmeiras,Fortaleza,2023-07-22,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...",0.2314,True,13,0.8,0.51570,6.016400,0.000000
6,7290221,Fezbet,over/under,3.5,over,4.50,0.222222,Bahia,Corinthians,2023-07-22,"[0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, ...",0.5144,True,13,0.6,0.55720,2.314800,0.000000
8,7290221,Fezbet,over/under,4.5,over,8.50,0.117647,Bahia,Corinthians,2023-07-22,"[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, ...",0.3640,True,13,0.7,0.53200,3.094000,0.000000
4,7290221,Stake.com,over/under,2.5,over,2.48,0.403226,Bahia,Corinthians,2023-07-22,"[0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, ...",0.6548,True,13,0.4,0.52740,1.623904,0.051325
12,7290221,1xBet,over/under,6.5,over,50.00,0.020000,Bahia,Corinthians,2023-07-22,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...",0.1503,True,13,0.9,0.52515,7.515000,0.000000
2,7290221,Stake.com,over/under,1.5,over,1.48,0.675676,Bahia,Corinthians,2023-07-22,"[0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, ...",0.8267,True,13,0.2,0.51335,1.223516,0.000000


In [12]:
odds_dt.solution

6     0.000000
4     0.000000
8     0.000000
20    0.000000
10    0.000000
6     0.000000
8     0.000000
4     0.000000
12    0.000000
2     0.821039
6     0.000000
8     0.000000
12    0.000000
4     0.000000
2     0.000000
10    0.000000
8     0.000000
6     0.178961
22    0.000000
19    0.000000
Name: solution, dtype: float64

In [87]:
solution

array([ 2.3594124 ,  0.46742427,  4.0703344 ,  0.90183145, -2.0635    ,
       -0.04187242, -1.9237189 ,  0.32340235,  0.05628312,  2.291533  ,
       -0.7601051 ,  3.616607  , -1.4812615 ,  0.30573812,  1.0348967 ,
       -0.5912484 ,  4.0191975 , -0.54406   , -5.3116217 , -0.42686996],
      dtype=float32)

In [80]:
solution

array([  3.897214  ,   1.4024419 ,   6.5735397 ,   1.085212  ,
         0.43374148, -10.        ,  -0.04147732,  -6.285349  ,
        -0.7573125 ,  -1.5447041 ,   4.692737  ,  -0.0546049 ,
        -4.6492615 ,  -1.4652156 ,  -3.2539277 ,   2.5623128 ,
         4.5390725 ,   1.2403467 ,  -1.806097  ,   6.364262  ],
      dtype=float32)

In [ ]:

def run_strategy(args):
    start_time = time()

    odds, gameid_to_outcome = setup(args)

    grouped = odds.groupby(args.aggregator)
    print(f"The number of jobs is: {args.n_jobs}")
    # Parallelize the group processing
    results = Parallel(n_jobs=args.n_jobs)(
        delayed(process_group)(group, gameid_to_outcome, args) for group in grouped
    )

    data = [x for x in results if x is not None]
    df_flat = pd.DataFrame([item for sublist in data for item in sublist])

    # # Start an MLflow experiment
    # with mlflow.start_run():
    #     # Log parameters (e.g., settings of the optimizer)
    #     mlflow.log_param("aggregator", args.aggregator)
    #     mlflow.log_param("min_games", args.min_games)
    #     mlflow.log_param("bookmakers", args.bookmakers)
    #     mlflow.log_param("bets_per_game", args.bets_per_game)
    #     mlflow.log_param("weight", args.weight)
    #     mlflow.log_param("do_baseline", args.do_baseline)
    #     mlflow.log_param("n_iterations", args.n_iterations)


    #     if args.save_experiment:
    #         # Create artefacts folder
    #         timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
    #         artefacts_folder = f"artefacts/{timestamp}"
    #         os.makedirs(artefacts_folder)
    #         save_csv_artifact(artefacts_folder, "result", df_flat)
    #         df_plot = build_plot_df_wrapper(artefacts_folder,  args.aggregator, args.do_baseline)
    #         save_csv_artifact(artefacts_folder, "result_plot", df_plot)
    #         save_plot_strategy(artefacts_folder, df_plot)
        
    #     mlflow.log_param("timestamp", timestamp)
    #     mlflow.log_artifact(f"{artefacts_folder}/result_plot.csv")
    #     mlflow.log_artifact(f"{artefacts_folder}/plot.PNG")
    #     mlflow.log_metric("wealth", df_plot["stake"].values[-1])

    #     # End the MLflow run
    #     mlflow.end_run()

    # elapsed_time = time() - start_time
    # print("Final Elapsed: %.3f sec" % elapsed_time)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        '--bookmakers',
        nargs='+',
        default=None,
        help='A list of strings',
    )
    parser.add_argument(
        "--aggregator", type=str, help="aggregate by GameId or by Datetime"
    )
    parser.add_argument(
        "--min_games",
        type=int,
        default=0,
        help="threshold of minimum number of games to enter the optimization task",
    )
    parser.add_argument(
        "--bets_per_game", type=int, default=5, help="number of bets per game"
    )
    parser.add_argument(
        "--weight",
        type=float,
        default=0.5,
        help="weight of the linear combination filter",
    )
    parser.add_argument(
        "--n_iterations",
        type=int,
        default=10,
        help="number of iterations to run the optimization task",
    )
    parser.add_argument(
        "--do_baseline",
        action="store_true",
        help="flag to apply baseline logic or not, not specifying the argument return the opposite of the action",
    )
    parser.add_argument(
        "--n_jobs",
        type=int,
        default=1,
        help="number of jobs to run in parallel",
    )
    parser.add_argument(
        "--save_experiment",
        action="store_true",
        help="flag to save the experiment artefacts, not specifying the argument return the opposite of the action",
    )
    args = parser.parse_args()
    print(args)
    run_strategy(args)
